# AutoAntibiotic — Single-Compound Screening (Jupyter Example)

This notebook shows how to screen a single SMILES string against MRSA PBP2a
using `discovery_pipeline` without generating a full library.

It is dependency-light: the `targets`/`deps` dicts are mocked so the example
runs without real PDBs. To enable real docking, set `USE_VINA=True` and supply
a prepared PBP2a receptor PDBQT plus grid centres (see `dp.prepare_targets`).

In [ ]:
# 1. Imports
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))  # make discovery_pipeline importable

import rdkit
from rdkit import Chem
from rdkit.Chem.Draw import MolToImage
import discovery_pipeline as dp
from discovery_pipeline import (
    screen_single_compound,
    analyze_binding_interactions,
    profile_resistance_risk,
)

print("RDKit", rdkit.__version__, "| discovery_pipeline loaded")

In [ ]:
# 2. Sample SMILES — a carbapenem (meropenem-like) PBP2a reference
SAMPLE_SMILES = (
    "CN1C(=O)C(N=C1C(=O)O)SC2=C(C3N(C2=O)C(=C(CS3)C(=O)O)"
    "C(=O)N(C4=CC=C(C=C4)N5CCCC5)C6=CC=C(C=C6)N7CCCC7)C(=O)O"
)

In [ ]:
# 3. Screen a single compound
# Normally `targets`/`deps` come from dp.prepare_targets(...) with real PDBs.
# Here we mock them so the example runs without external binaries. Set
# USE_VINA=True and provide real PBP2a PDBQT paths + grid centres to dock.
deps = {"vina": False, "USE_VINA": False}
targets = {
    "PBP2a": {
        "pdbqt": "/path/to/PBP2a.pdbqt",      # prepared receptor PDBQT
        "cleaned_pdb": "/path/to/PBP2a_clean.pdb",
        "allosteric_center": None,
        "active_center": None,
    }
}

rec = screen_single_compound(SAMPLE_SMILES, targets, work_dir=".", deps=deps)
print("Compound ID :", rec.compound_id)
print("Allosteric  :", rec.pb2pa_allosteric_energy)
print("Active      :", rec.pb2pa_active_energy)

In [ ]:
# 4. Display the 2D structure
mol = Chem.MolFromSmiles(SAMPLE_SMILES)
img = MolToImage(mol, size=(400, 400))
img

In [ ]:
# 5. Interaction summary (requires a docked active-site pose)
pose = getattr(rec, "active_docked_pdbqt", None)
cleaned = targets["PBP2a"].get("cleaned_pdb")
if pose and cleaned:
    interactions = analyze_binding_interactions(pose, cleaned)
    rec.interactions = interactions
    rec.resistance_notes = profile_resistance_risk(
        rec, ".", targets["PBP2a"]["pdbqt"],
        targets["PBP2a"]["active_center"], dp.ACTIVE_BOX_SIZE,
        interactions=interactions,
    )
else:
    rec.resistance_notes = profile_resistance_risk(
        rec, ".", targets["PBP2a"]["pdbqt"],
        targets["PBP2a"]["active_center"], dp.ACTIVE_BOX_SIZE,
    )

print("Interaction Summary:")
print(" ", rec.resistance_notes)